# Convert NPY Shards to Parquet

Конвертация npy-шардов активаций Gemma-3 в Parquet с ZSTD-сжатием.
Объединяет по 10 шардов в один Parquet-файл и пушит на HuggingFace Hub.

**Пиковая RAM:** ~7.5 ГБ (10 шардов × 768 МБ в float32)

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))

In [ ]:
import os
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfApi
from tqdm import tqdm

REPO = "veriga/openwebtext-gemma3-tokenized-1024-activations-layer23"
SHARDS_PER_PARQUET = 10
COMPRESSION = "zstd"
COMPRESSION_LEVEL = 3
TOTAL_SHARDS = 1121  # 000000–001120

api = HfApi()
LOCAL_DIR = "./shards"
os.makedirs(LOCAL_DIR, exist_ok=True)

In [ ]:
for batch_start in tqdm(range(0, TOTAL_SHARDS, SHARDS_PER_PARQUET), desc="Batches"):
    batch_end = min(batch_start + SHARDS_PER_PARQUET, TOTAL_SHARDS)
    shard_ids = [f"{i:06d}" for i in range(batch_start, batch_end)]

    all_act, all_mask, all_tok = [], [], []

    # Скачиваем шарды
    for sid in shard_ids:
        for suffix in ["", "_masks", "_tokens"]:
            fname = f"shard_{sid}{suffix}.npy"
            api.hf_hub_download(
                repo_id=REPO,
                filename=fname,
                repo_type="dataset",
                local_dir=LOCAL_DIR,
            )

    # Загружаем в память
    for sid in shard_ids:
        act = np.load(f"{LOCAL_DIR}/shard_{sid}.npy").astype(np.float32)
        mask = np.load(f"{LOCAL_DIR}/shard_{sid}_masks.npy")
        tok = np.load(f"{LOCAL_DIR}/shard_{sid}_tokens.npy")
        all_act.append(act)
        all_mask.append(mask)
        all_tok.append(tok)

    act_cat = np.concatenate(all_act, axis=0)
    mask_cat = np.concatenate(all_mask, axis=0)
    tok_cat = np.concatenate(all_tok, axis=0)

    # Чистим npy
    for sid in shard_ids:
        for suffix in ["", "_masks", "_tokens"]:
            os.remove(f"{LOCAL_DIR}/shard_{sid}{suffix}.npy")

    N = act_cat.shape[0]

    # Конвертим в Parquet
    table = pa.table({
        "activations": [act_cat[i] for i in range(N)],
        "mask": [mask_cat[i] for i in range(N)],
        "tokens": [tok_cat[i] for i in range(N)],
    })

    parquet_name = f"train-{batch_start:06d}-{batch_end-1:06d}.parquet"
    parquet_path = f"./{parquet_name}"
    pq.write_table(table, parquet_path, compression=COMPRESSION, compression_level=COMPRESSION_LEVEL)

    size_mb = os.path.getsize(parquet_path) / 1e6
    del act_cat, mask_cat, tok_cat, all_act, all_mask, all_tok, table

    # Пушим на Hub
    api.upload_file(
        path_or_fileobj=parquet_path,
        path_in_repo=f"data/{parquet_name}",
        repo_id=REPO,
        repo_type="dataset",
        commit_message=f"Add parquet batch {batch_start}-{batch_end-1}",
    )

    os.remove(parquet_path)
    tqdm.write(f"  Batch {batch_start}-{batch_end-1}: {N} examples, {size_mb:.0f} MB")

print(f"Done! {TOTAL_SHARDS} shards -> {len(range(0, TOTAL_SHARDS, SHARDS_PER_PARQUET))} parquet files in data/")